# BigQuery starter

<a href="https://colab.research.google.com/github/openactive-contrib/openactive-admin-dashboard/blob/main/notebooks/01_bigquery_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Signs in to BigQuery with your own Google account and reads two small tables from
`openactive-monitor.openactive_analytics`: how many publishers, datasets and feeds are in the
catalogue, and how the fleet's future opportunities have moved over recent insight runs. Each
query scans well under 1 GB.

Your Google account needs read access to the dataset and permission to run query jobs in the
`openactive-monitor` project. If a query fails with *Access Denied*, ask the tech team to
grant it.

In [ ]:
import sys

from google.cloud import bigquery

PROJECT = "openactive-monitor"
DATASET = "openactive-monitor.openactive_analytics"
MAX_GB = 5  # refuse any single query that would scan more than this

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()  # opens a Google sign-in; queries then run as your account

client = bigquery.Client(project=PROJECT)


def run_query(sql, params=None, max_gb=MAX_GB):
    """Dry-run, report the scan size, then run with a hard cap on bytes billed."""
    config = bigquery.QueryJobConfig(query_parameters=params or [])
    dry = client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=params or [], dry_run=True, use_query_cache=False
        ),
    )
    gb = dry.total_bytes_processed / 1e9
    print(f"This query will scan {gb:.2f} GB.")
    if gb > max_gb:
        raise RuntimeError(f"Scan of {gb:.2f} GB exceeds max_gb={max_gb}; narrow the query.")
    config.maximum_bytes_billed = int(max_gb * 1e9)
    return client.query(sql, job_config=config).to_dataframe()

In [ ]:
DAYS = 60  # @param {type:"integer"}

## What is in the catalogue

Publishers, datasets and feeds, split into regular feeds (the public catalogue) and preview
feeds. `feeds` can hold several rows per feed, so only the latest row per feed is counted.

In [ ]:
sql_catalogue = f"""
WITH latest_feeds AS (
  SELECT id, dataset_url, publisher_name, is_regular
  FROM `{DATASET}.feeds`
  QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY last_access DESC) = 1
)
SELECT
  IF(is_regular, 'regular', 'preview') AS catalogue,
  COUNT(DISTINCT publisher_name) AS publishers,
  COUNT(DISTINCT dataset_url) AS datasets,
  COUNT(*) AS feeds
FROM latest_feeds
GROUP BY catalogue
ORDER BY catalogue DESC
"""
catalogue = run_query(sql_catalogue)
display(catalogue)

## Future opportunities over recent runs

One row per run of the insight job, from `insight_run_summary`. "Future" means a start date
from midnight UTC on the run's day onwards. The figure counts opportunity items in regular
feeds.

In [ ]:
import plotly.express as px

sql_runs = f"""
SELECT
  run_date,
  num_publishers_regular,
  total_num_future_opportunity_items_regular
FROM `{DATASET}.insight_run_summary`
WHERE run_date >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL @days DAY)
ORDER BY run_date
"""
runs = run_query(sql_runs, [bigquery.ScalarQueryParameter("days", "INT64", DAYS)])

fig = px.line(
    runs,
    x="run_date",
    y="total_num_future_opportunity_items_regular",
    markers=True,
    title=f"Future opportunity items in regular feeds, last {DAYS} days",
    labels={
        "run_date": "Insight run (UTC)",
        "total_num_future_opportunity_items_regular": "Future opportunity items",
    },
)
fig.show()

## Notes

- The tables are refreshed by a daily batch. Nothing here is live.
- Counts follow the definitions of the pipeline's insight job
  (`jobs/opportunity-insights/queries.py` in `openactive-contrib/openactive-monitor`).
- To write your own notebook with an LLM, give it the skills in `notebooks/skills/`.